In [ ]:
from openai import OpenAI
import os
from google.colab import userdata
from tqdm.notebook import tqdm

!rm -rf Classical-Chinese-Summarization
!git clone https://github.com/ctaiyi15/Classical-Chinese-Summarization.git



client = OpenAI(
    api_key=userdata.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)


import re

def count_paragraphs(text):
    paragraphs = re.split(r"\n\s*\n", text.strip())
    return len([p for p in paragraphs if p.strip()])

def count_words(text):
    return len(text.split())

def get_word_minmax(source_word_count):
  if 1000 <= source_word_count < 4000:
      return 200, 400
  elif 4000 <= source_word_count < 10000:
      return 400, 600
  elif source_word_count >= 10000:
      return 600, 800
  else:
      return None, None  # skipped case if source_word_count < 1000

def get_p_minmax(source_word_count):
  if 1000 <= source_word_count < 4000:
      return 1, 4
  elif 4000 <= source_word_count < 10000:
      return 2, 6
  elif source_word_count >= 10000:
      return 3, 8
  else:
      return None, None  # skipped case if source_word_count < 1000

# Check if a summary satisfies word and paragraph limits given by the prompt
def validate_summary(summary, source_word_count):
    word_count = count_words(summary)
    paragraph_count = count_paragraphs(summary)
    word_min, word_max = get_word_minmax(source_word_count)
    para_min, para_max = get_p_minmax(source_word_count)
    if word_min is None or para_min is None:
      return None
    return {
        "word_count": word_count,
        "paragraph_count": paragraph_count,
        "word_ok": word_min <= word_count <= word_max,
        "paragraph_ok": para_min <= paragraph_count <= para_max
    }

def summarize(text):
    word_count = len(text.split())

    # Skip short texts
    if word_count < 1000:
        return None

    # Get required word and paragraph limits for prompts
    word_min, word_max = get_word_minmax(word_count)
    p_min, p_max = get_p_minmax(word_count)


    prompt = f"""
Summarize the following translated historical text.

CRITICAL RULES:
Length Control: - The summary MUST be between {word_min} and {word_max} words.
- If the summary is longer than {word_max} words, condense it by stripping away unimportant information until it contains words no more than the maximum word count.
- The summary should contain {p_min}-{p_max} paragraphs.
Strict Faithfulness: Rely STRICTLY on the information provided in the text. DO NOT add any outside historical knowledge, dates, or names to pad the length.
No Inference: If the text lacks context, do not attempt to fill in the blanks.

TEXT TO SUMMARIZE:
{text}
"""

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role": "system",
                "content": "You are a meticulous historical archivist specializing in Classical Chinese texts. Your task is to accurately condense translated historical records into English without altering facts."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


from pathlib import Path

input_root = Path("Classical-Chinese-Summarization/data/processed/translated")
output_root = Path("Classical-Chinese-Summarization/data/output/translated")

files = list(input_root.rglob("*.txt"))

total = 0
violations = 0
word_violations = 0
para_violations = 0

for file_path in tqdm(files):
    # get relative path
    relative_path = file_path.relative_to(input_root)

    # build output path
    output_path = output_root / relative_path

    # change extension
    output_path = output_path.with_suffix(".summary.txt")

    # create folders if needed
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # read input
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        continue

    result = summarize(text)

    if result is None:
        continue

    # Save result
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(result)

    # Update violation stats
    total += 1

    source_wc = len(text.split())

    check = validate_summary(result, source_wc)

    if not check["word_ok"] or not check["paragraph_ok"]:
      violations += 1
      if not check["word_ok"]:
        word_violations += 1
      if not check["paragraph_ok"]:
        para_violations += 1

    print("⚠️ Constraint violation:", output_path)
    print(check, f"length:{source_wc}")

    print("Saved:", output_path)

# Print violation stats
if total > 0:
    print("\n=== SUMMARY STATS ===")
    print(f"Total processed: {total}")
    print(f"Total violations: {violations} ({violations/total:.2%})")
    print(f"Word count violations: {word_violations} ({word_violations/total:.2%})")
    print(f"Paragraph violations: {para_violations} ({para_violations/total:.2%})")



Cloning into 'Classical-Chinese-Summarization'...
remote: Enumerating objects: 1799, done.
remote: Counting objects: 100% (1799/1799), done.
remote: Compressing objects: 100% (947/947), done.
remote: Total 1799 (delta 40), reused 1770 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (1799/1799), 11.91 MiB | 15.10 MiB/s, done.
Resolving deltas: 100% (40/40), done.


  0%|          | 0/411 [00:00<?, ?it/s]

⚠️ Constraint violation: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第十章/target.summary.txt
{'word_count': 538, 'paragraph_count': 7, 'word_ok': True, 'paragraph_ok': False} length:8240
Saved: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第十章/target.summary.txt
⚠️ Constraint violation: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第三章/target.summary.txt
{'word_count': 406, 'paragraph_count': 4, 'word_ok': False, 'paragraph_ok': True} length:14888
Saved: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第三章/target.summary.txt
⚠️ Constraint violation: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第八章/target.summary.txt
{'word_count': 377, 'paragraph_count': 4, 'word_ok': False, 'paragraph_ok': True} length:9381
Saved: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第八章/target.summary.txt
⚠️ Constraint violation: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第九章/target.summary.txt
{'word_c

In [ ]:
!zip -r summaries.zip Classical-Chinese-Summarization/data/output/translated

  adding: Classical-Chinese-Summarization/data/output/translated/ (stored 0%)
  adding: Classical-Chinese-Summarization/data/output/translated/晋书/ (stored 0%)
  adding: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/ (stored 0%)
  adding: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第十章/ (stored 0%)
  adding: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第十章/target.summary.txt (deflated 54%)
  adding: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第三章/ (stored 0%)
  adding: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第三章/target.summary.txt (deflated 52%)
  adding: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第八章/ (stored 0%)
  adding: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第八章/target.summary.txt (deflated 53%)
  adding: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第九章/ (stored 0%)
  adding: Classical-Chinese-Summarization/data/output/translated/晋书/帝纪/第九章/tar

In [ ]:
from google.colab import files

files.download("summaries.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>